# FT09 — prompt experiments
 Change and inspect prompts here. Runs go to `experiments/`, never the training collection.
 Game functions, the default prompt and parser are imported from `ft09_shared.py`.

 The earlier saved experiments remain in `Exploring_ARC_AGI_SDK_FT09.ipynb` and `PROMPT_EXPERIMENTS.md`.


## Dependencies
Run this cell first. If packages were installed or upgraded, restart the kernel, then continue with the cells below.


In [ ]:
%pip install -q "arc-agi==0.9.9" numpy pillow matplotlib pyarrow openai

In [ ]:
from pathlib import Path
import sys

# Open this notebook from the project folder containing ft09_shared.py.
PROJECT_DIR = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "ft09_shared.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    raise RuntimeError("Open the notebook from the FT09 project folder.")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import ft09_shared as ft
ft.configure_project(PROJECT_DIR)
print("Project:", PROJECT_DIR)


## 1. Set the experiment


In [ ]:
PROMPT = ft.GAME_PROMPT
LEVEL = 1
N_ATTEMPTS = 1
MAX_STEPS = 15
TEMPERATURE = 1.0
print(PROMPT)


## 2. Connect the model
This cell asks for the API key privately. It does not save the key.


In [ ]:
from getpass import getpass
from ft09_models import OpenAIBackend

MODEL = "lisa-pro"
BASE_URL = "https://chat-1.ki-awz.iisys.de/api"
if "backend" in globals():
    backend.close()
api_key = getpass("LISA API key: ")
backend = OpenAIBackend(MODEL, api_key, BASE_URL, timeout=180, max_new_tokens=512)
del api_key


## 3. Start a new output folder
Run this cell again for a new experiment. The prompt and settings are saved with its results.


In [ ]:
RUN_DIR = ft.new_run_dir("experiments", MODEL)
backend.log_path = RUN_DIR / "requests.jsonl"
ft.save_run_config(
    RUN_DIR, backend, PROMPT, phase="prompt_experiment", levels=[LEVEL],
    attempts=N_ATTEMPTS, temperature=TEMPERATURE, max_steps=MAX_STEPS,
)
print("Output:", RUN_DIR)


## 4. Run


In [ ]:
records = ft.run_level(
    backend, LEVEL, RUN_DIR, n_attempts=N_ATTEMPTS,
    temperature=TEMPERATURE, max_steps=MAX_STEPS,
    prompt=PROMPT, show_frames=True,
)


## 5. Summarise


In [ ]:
attempt_results, level_summary = ft.summarise_evaluation(RUN_DIR)
